RAG CoT
(Chain of Thought)

RAG 파이프라인에서 LLM이 단순한 정보 조합을 넘어서 단계적 사고를 통해 논리적 답변을 할 수 있도록 한다.

In [3]:
%pip install langchain langchain-openai -Uq


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_API_KEY'] = os.getenv('langsmith_key')
os.environ['LANGSMITH_PROJECT'] = 'skn23-langchain'
os.environ['OPENAI_API_KEY'] = os.getenv("openai_key")
os.environ['PINECONE_API_KEY'] = os.getenv("pinecone_key")
os.environ['COHERE_API_KEY'] = os.getenv('cohere_key')

# 가상 Retriever

In [ ]:
from langchain_core.documents import Document

def retrieve_vectordb(query = None):
    return [
        Document(page_content="대한민국의 수도는 서울입니다. 서울은 한강을 끼고 발달한 도시입니다."),  # 서울 기본 정보
        Document(page_content="서울의 대표적 관광지는 경복궁, 남산타워, 명동 등이 있습니다."),  # 서울 관광 정보
        Document(page_content="서울의 인구는 약 천만 명이며, 교통·문화 인프라가 잘 갖춰져 있습니다.")  # 서울 인구/인프라 정보
    ]
    
retrieve_vectordb()
    

[Document(metadata={}, page_content='대한민국의 수도는 서울입니다. 서울은 한강을 끼고 발달한 도시입니다.'),
 Document(metadata={}, page_content='서울의 대표적 관광지는 경복궁, 남산타워, 명동 등이 있습니다.'),
 Document(metadata={}, page_content='서울의 인구는 약 천만 명이며, 교통·문화 인프라가 잘 갖춰져 있습니다.')]

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.chat_models import init_chat_model

prompt = PromptTemplate.from_template('''  # 검색 문서 기반 답변 지시 프롬프트
당신은 데이터를 분석해서 논리적인 결론을 도출하는 전문가 챗봇입니다.
아래 [검색된 문서]를 바탕으로 사용자의 [질문]에 대해 답변하세요.

[검색된 문서]
{context}

[질문]
{question}

[지시사항]
다음의 단계에 따라 사고한 후, 답변을 작성하세요.
1. **핵심데이터 정리**: 문서에서 사용자 질문과 관련한 팩트를 추출해보세요.
2. **상호관계 분석**: 각 항목별로 어떤 상관관계/시너지를 도출하는지 고민하세요.
3. **논리적 서술**: 위의 사고한 내용을 토대로 사용자 질문에 대한 답변을 준비하세요.
4. **최종 답변**: 서론-본론-결론 구조에 맞춰 완성된 답변을 작성하세요.
''')

llm = init_chat_model('gpt-4.1-mini')
output_parser = StrOutputParser()

question = '서울의 인구, 관광지, 교육인프라를 종합해서 여행하기 좋은 이유를 논리적으로 설명해주세요.'

retrieved_docs = retrieve_vectordb()

retrieved_docs

context = "\n\n".join(doc.page_content for doc in retrieved_docs)           # page_content만 합치기

chain = prompt | llm | output_parser

response =  chain.invoke({'context':context,'question':question})           # 컨텍스트 + 질문으로 답변 생성
print(response)

'1. **핵심데이터 정리**  \n- 서울의 인구: 약 천만 명  \n- 서울의 관광지: 경복궁, 남산타워, 명동 등  \n- 서울의 교육 인프라: 문서에는 직접 언급되어 있지 않으나, 서울은 교통·문화 인프라가 잘 갖춰져 있다고 명시됨  \n\n2. **상호관계 분석**  \n- 인구가 약 천만 명이라는 점은 서울이 대도시로서 다양한 문화와 서비스가 풍부함을 의미한다.  \n- 대표적인 관광지가 경복궁, 남산타워, 명동 등 다양해 여행자의 다양한 취향을 만족시킬 수 있다.  \n- 교통 인프라가 잘 갖춰져 있어 관광지 접근이 용이하며, 문화 인프라가 발달해 여행 중 다양한 체험과 편의를 제공한다.  \n- 방대한 인구와 발달된 인프라가 서로 시너지를 이루어 여행객에게 편리하고 풍부한 여행 경험을 제공한다.  \n\n3. **논리적 서술**  \n서울은 약 천만 명의 인구가 거주하는 대도시로서, 다양한 문화적 배경과 서비스가 공존한다. 이러한 인구 규모는 관광과 교육, 문화 인프라의 발전을 촉진하며, 이는 다시 관광객에게 다양한 선택지와 체험 기회를 제공한다. 경복궁, 남산타워, 명동과 같은 대표 관광지는 서울의 역사와 현대문화를 동시에 경험할 수 있게 해 주며, 잘 발달된 교통 인프라는 여행의 편리함을 높여준다. 비록 교육 인프라에 대한 구체적 언급은 없으나, 대도시 서울의 특성상 우수한 교육기관과 학습 환경이 갖춰져 있어 교육과 관련된 다양한 정보와 체험 기회도 제공 가능하다.  \n\n4. **최종 답변**  \n서울은 약 천만 명에 달하는 인구를 바탕으로 풍부한 문화와 다양한 서비스가 공존하는 대도시입니다. 이로 인해 경복궁, 남산타워, 명동 등 역사적·현대적 매력을 지닌 관광지가 풍부하며, 동시에 서울 전역에 잘 발달된 교통과 문화 인프라가 갖춰져 있어 여행객이 쉽게 다양한 장소를 방문하고 체험할 수 있습니다. 또한, 대도시로서 우수한 교육 환경 역시 갖추고 있기 때문에 관광뿐 아니라 교육적인 측면에서도 깊이 있는 경험을 할 수 있다는 점이 서울을 

In [17]:
print(response)

1. **핵심데이터 정리**  
- 서울의 인구: 약 천만 명  
- 서울의 관광지: 경복궁, 남산타워, 명동 등  
- 서울의 교육 인프라: 문서에는 직접 언급되어 있지 않으나, 서울은 교통·문화 인프라가 잘 갖춰져 있다고 명시됨  

2. **상호관계 분석**  
- 인구가 약 천만 명이라는 점은 서울이 대도시로서 다양한 문화와 서비스가 풍부함을 의미한다.  
- 대표적인 관광지가 경복궁, 남산타워, 명동 등 다양해 여행자의 다양한 취향을 만족시킬 수 있다.  
- 교통 인프라가 잘 갖춰져 있어 관광지 접근이 용이하며, 문화 인프라가 발달해 여행 중 다양한 체험과 편의를 제공한다.  
- 방대한 인구와 발달된 인프라가 서로 시너지를 이루어 여행객에게 편리하고 풍부한 여행 경험을 제공한다.  

3. **논리적 서술**  
서울은 약 천만 명의 인구가 거주하는 대도시로서, 다양한 문화적 배경과 서비스가 공존한다. 이러한 인구 규모는 관광과 교육, 문화 인프라의 발전을 촉진하며, 이는 다시 관광객에게 다양한 선택지와 체험 기회를 제공한다. 경복궁, 남산타워, 명동과 같은 대표 관광지는 서울의 역사와 현대문화를 동시에 경험할 수 있게 해 주며, 잘 발달된 교통 인프라는 여행의 편리함을 높여준다. 비록 교육 인프라에 대한 구체적 언급은 없으나, 대도시 서울의 특성상 우수한 교육기관과 학습 환경이 갖춰져 있어 교육과 관련된 다양한 정보와 체험 기회도 제공 가능하다.  

4. **최종 답변**  
서울은 약 천만 명에 달하는 인구를 바탕으로 풍부한 문화와 다양한 서비스가 공존하는 대도시입니다. 이로 인해 경복궁, 남산타워, 명동 등 역사적·현대적 매력을 지닌 관광지가 풍부하며, 동시에 서울 전역에 잘 발달된 교통과 문화 인프라가 갖춰져 있어 여행객이 쉽게 다양한 장소를 방문하고 체험할 수 있습니다. 또한, 대도시로서 우수한 교육 환경 역시 갖추고 있기 때문에 관광뿐 아니라 교육적인 측면에서도 깊이 있는 경험을 할 수 있다는 점이 서울을 여행하기 좋은 도시로 만드는 